# Importing Libraries and Datasets

In [577]:
import pandas as pd
import re
from rapidfuzz import process
import nltk

In [578]:
df_1 = pd.read_csv("../datasets/Audible_Catlog.csv")
df_1

,Book Name,Author,Rating,Number of Reviews,Price
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,313.0,10080.0
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3658.0,615.0
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20174.0,10378.0
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,4614.0,888.0
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4.6,4302.0,1005.0
...,...,...,...,...,...
6363,The Hot Flash Club,Nancy Thayer,4.3,191.0,1131.0
6364,The Prophet & The Wanderer,Khalil Gibran,4.1,6.0,539.0
6365,Make Today Count: The Secret of Your Success I...,John C. Maxwell,4.7,301.0,500.0
6366,25 Hours a Day: Going One More to Get What You...,Nick Bare,4.7,285.0,501.0


In [579]:
df_2 = pd.read_csv("../datasets/Audible_Catlog_Advanced_Features.csv")
df_2

,Book Name,Author,Rating,Number of Reviews,Price,Description,Listening Time,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,371.0,10080,"Over the past three years, Jay Shetty has beco...",10 hours and 54 minutes,",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3682.0,615,Brought to you by Penguin.,3 hours and 23 minutes,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20306.0,10378,"In this generation-defining self-help guide, a...",5 hours and 17 minutes,",#3 in Audible Audiobooks & Originals (See Top..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,4678.0,888,Brought to you by Penguin.,5 hours and 35 minutes,",#5 in Audible Audiobooks & Originals (See Top..."
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4.6,4308.0,1005,"Stop going through life, Start growing throug...",6 hours and 25 minutes,",#6 in Audible Audiobooks & Originals (See Top..."
...,...,...,...,...,...,...,...,...
4459,"Factfulness: Wie wir lernen, die Welt so zu se...",Hans Rosling,4.6,72.0,703,"Sorry, we just need to make sure you're not a ...",-1,-1
4460,Late-Talking Children: A Symptom or a Stage?,Stephen M. Camarata,4.6,92.0,703,"Sorry, we just need to make sure you're not a ...",-1,-1
4461,"The Marketing of Evil: How Radicals, Elitists ...",David Kupelian,4.7,490.0,586,"Americans have come to tolerate, embrace, and ...",-1,-1
4462,Things I Wish I'd Known Before We Got Married,Gary Chapman,4.7,1388.0,516,\n\nOops!\nIt's rush hour and traffic is pilin...,-1,-1


### Merging datasets together

In [580]:
df_merged = df_1.merge(df_2, on='Book Name', how='outer')
df_merged.drop_duplicates('Book Name', inplace=True)

In [581]:
# Reset Index
df_merged.reset_index(inplace=True)
df_merged.drop(columns=['index'], inplace=True)

# Data Preprocessing

## Data Cleaning

In [582]:
# Basic Info
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6054 entries, 0 to 6053
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Book Name            6054 non-null   object 
 1   Author_x             5396 non-null   object 
 2   Rating_x             5396 non-null   float64
 3   Number of Reviews_x  4874 non-null   float64
 4   Price_x              5394 non-null   float64
 5   Author_y             4007 non-null   object 
 6   Rating_y             4007 non-null   float64
 7   Number of Reviews_y  3630 non-null   float64
 8   Price_y              4007 non-null   float64
 9   Description          4001 non-null   object 
 10  Listening Time       4007 non-null   object 
 11  Ranks and Genre      4007 non-null   object 
dtypes: float64(6), object(6)
memory usage: 567.7+ KB


In [583]:
df_merged.describe(include='all')

,Book Name,Author_x,Rating_x,Number of Reviews_x,Price_x,Author_y,Rating_y,Number of Reviews_y,Price_y,Description,Listening Time,Ranks and Genre
count,6054,5396,5396.000000,4874.000000,5394.000000,4007,4007.000000,3630.000000,4007.000000,4001,4007,4007
unique,6054,3537,NaN,NaN,NaN,2693,NaN,NaN,NaN,2320,1021,2026
top,"""Don't You Know Who I Am?"": How to Stay Sane i...",Harvard Business Review,NaN,NaN,NaN,Harvard Business Review,NaN,NaN,NaN,"Sorry, we just need to make sure you're not a ...",-1,-1
freq,1,32,NaN,NaN,NaN,24,NaN,NaN,NaN,962,1964,1964
mean,NaN,NaN,3.928002,966.802011,931.211531,NaN,3.941527,1078.129201,960.063888,NaN,NaN,NaN
std,NaN,NaN,1.647055,2612.621093,1575.939982,NaN,1.627711,2888.341932,1673.806853,NaN,NaN,NaN
min,NaN,NaN,-1.000000,1.000000,0.000000,NaN,-1.000000,1.000000,0.000000,NaN,NaN,NaN
25%,NaN,NaN,4.200000,67.000000,501.000000,NaN,4.200000,72.250000,501.000000,NaN,NaN,NaN
50%,NaN,NaN,4.500000,242.000000,683.000000,NaN,4.500000,267.500000,683.000000,NaN,NaN,NaN
75%,NaN,NaN,4.600000,798.000000,888.000000,NaN,4.600000,909.000000,888.000000,NaN,NaN,NaN


### Percentage of missing values

In [584]:
def per_miss_val(df):
  return 100 * df.isnull().sum() / len(df)

per_miss_val(df_merged)

Book Name               0.000000
Author_x               10.868847
Rating_x               10.868847
Number of Reviews_x    19.491245
Price_x                10.901883
Author_y               33.812355
Rating_y               33.812355
Number of Reviews_y    40.039643
Price_y                33.812355
Description            33.911463
Listening Time         33.812355
Ranks and Genre        33.812355
dtype: float64

### Filling missing values with existing values

In [585]:
# Author Names
df_merged['Author_x'] = df_merged['Author_x'].fillna(df_merged['Author_y'])

In [586]:
# Ratings
df_merged['Rating_x'] = df_merged['Rating_x'].fillna(df_merged['Rating_y'])

In [587]:
# No. of Reviews
df_merged['Number of Reviews_x'] = df_merged['Number of Reviews_x'].fillna(df_merged['Number of Reviews_y'])

In [588]:
# Price
df_merged['Price_x'] = df_merged['Price_x'].fillna(df_merged['Price_y'])

### Removing duplicate records

In [589]:
df_merged.drop(columns=['Author_y', 'Rating_y', 'Number of Reviews_y', 'Price_y'], inplace=True)

In [590]:
per_miss_val(df_merged)

Book Name               0.000000
Author_x                0.000000
Rating_x                0.000000
Number of Reviews_x     9.580443
Price_x                 0.016518
Description            33.911463
Listening Time         33.812355
Ranks and Genre        33.812355
dtype: float64

### Dropping rows with missing values

In [591]:
df_merged.dropna(subset=['Number of Reviews_x', 'Price_x', 'Description', 'Listening Time', 'Ranks and Genre'], inplace=True)

In [592]:
# Missing value percentage
per_miss_val(df_merged)

Book Name              0.0
Author_x               0.0
Rating_x               0.0
Number of Reviews_x    0.0
Price_x                0.0
Description            0.0
Listening Time         0.0
Ranks and Genre        0.0
dtype: float64

### Checking for Data Integrity and Consistency

In [593]:
df_merged.describe(include='all')

,Book Name,Author_x,Rating_x,Number of Reviews_x,Price_x,Description,Listening Time,Ranks and Genre
count,3625,3625,3625.000000,3625.000000,3625.000000,3625,3625,3625
unique,3625,2493,NaN,NaN,NaN,2120,972,1851
top,#Girlboss,Devdutt Pattanaik,NaN,NaN,NaN,"Sorry, we just need to make sure you're not a ...",-1,-1
freq,1,24,NaN,NaN,NaN,854,1770,1770
mean,NaN,NaN,4.453352,1070.810207,987.195034,NaN,NaN,NaN
std,NaN,NaN,0.363080,2878.321320,1742.960459,NaN,NaN,NaN
min,NaN,NaN,-1.000000,1.000000,0.000000,NaN,NaN,NaN
25%,NaN,NaN,4.300000,71.000000,512.000000,NaN,NaN,NaN
50%,NaN,NaN,4.500000,265.000000,683.000000,NaN,NaN,NaN
75%,NaN,NaN,4.700000,903.000000,888.000000,NaN,NaN,NaN


### Rating Column

Rows with `-1` represents that the book does not have any ratings. Hence removing rows with value `-1`.

In [594]:
df_merged = df_merged[df_merged['Rating_x'] != -1]

In [595]:
df_merged['Rating_x'].describe()

count    3624.000000
mean        4.454857
std         0.351643
min         1.000000
25%         4.300000
50%         4.500000
75%         4.700000
max         5.000000
Name: Rating_x, dtype: float64

### Ranks and Genre Column

Removing rows with `-1` value.

In [596]:
df_merged['Ranks and Genre'].describe()

count     3624
unique    1851
top         -1
freq      1769
Name: Ranks and Genre, dtype: object

In [597]:
df_merged = df_merged[df_merged['Ranks and Genre'] != '-1']
df_merged.reset_index(inplace=True)
df_merged.drop(columns=['index'], inplace=True)

In [598]:
df_merged['Ranks and Genre'].describe()

count                                                  1855
unique                                                 1850
top       ,            5 star,          ,            ,  ...
freq                                                      3
Name: Ranks and Genre, dtype: object

Removing rows with inconsistent/incorrect values

In [599]:
# Removing incorrent data rows
def remove_inconsistent_values(val):
    val_list = [x for x in val.split("#") if x != ',']
    for i in val_list:
        if "%" in i:
            return None
    return val_list

df_merged['Ranks and Genre'] = df_merged['Ranks and Genre'].apply(lambda x: remove_inconsistent_values(x))
df_merged.dropna(subset=['Ranks and Genre'], inplace=True)

### Extracting Audible rank, genre and genre ranks

In [600]:
def extract_ranks(rank_list):
    overall_rank = None
    genre_ranks = {}

    for item in rank_list:
        match = re.search(r'(\d{1,3}(?:,\d{3})*)\s+in\s+(.+?)(?:\(|,|$)', item)
        
        if match:
            # Remove commas before converting to int
            rank = int(match.group(1).replace(',', ''))
            category = match.group(2).strip()

            if "Audible Audiobooks & Originals" in category:
                overall_rank = rank
            else:
                genre_ranks[category] = rank

    return overall_rank, genre_ranks

# Storing audible rank in a column, storing genre and its ranks inside a dictionary 
df_merged[['Audible_Rank', 'genre_ranks']] = df_merged['Ranks and Genre'].apply(
    lambda x: pd.Series(extract_ranks(x))
)

In [601]:
# Looking for books which is unranked in Audible
per_miss_val(df_merged)

Book Name              0.00000
Author_x               0.00000
Rating_x               0.00000
Number of Reviews_x    0.00000
Price_x                0.00000
Description            0.00000
Listening Time         0.00000
Ranks and Genre        0.00000
Audible_Rank           3.86921
genre_ranks            0.00000
dtype: float64

Filling missing values in `Audible_Rank` with `-1`

In [602]:
df_merged['Audible_Rank'] = df_merged['Audible_Rank'].fillna(-1)

In [603]:
df_merged

,Book Name,Author_x,Rating_x,Number of Reviews_x,Price_x,Description,Listening Time,Ranks and Genre,Audible_Rank,genre_ranks
0,10 Essential Success Mantras from the Bhagavad...,Vimla Patil,4.2,45.0,233.0,"For thousands of years, the Bhagavad Gita has ...",26 minutes,"[3,076 in Audible Audiobooks & Originals (See ...",3076.0,"{'Hinduism': 45, 'Forecasting & Strategic Plan..."
1,10 Masterpieces You Have to Read Before You Die 1,Jane Austen,4.4,331.0,401.0,This Audiobook contains the following works :,102 hours and 19 minutes,"[1,485 in Audible Audiobooks & Originals (See ...",1485.0,"{'Caribbean & Latin American Literature': 4, '..."
2,10 Minutes 38 Seconds in This Strange World,Elif Shafak,4.5,507.0,752.0,Shortlisted for the Booker Prize 2019,9 hours and 11 minutes,[221 in Audible Audiobooks & Originals (See To...,221.0,"{'Friendship Fiction': 1, 'Women's Fiction': 3..."
3,100 Baggers: Stocks That Return 100-to-1 and H...,Christopher W. Mayer,4.5,178.0,668.0,This book is about 100-baggers. These are stoc...,6 hours and 19 minutes,"[2,004 in Audible Audiobooks & Originals (See ...",2004.0,"{'Money & Finance': 184, 'Business & Economics..."
4,1000+ Little Things Happy Successful People Do...,Marc Chernoff,4.7,210.0,1005.0,New York Times best-selling authors Marc and A...,8 hours and 50 minutes,[719 in Audible Audiobooks & Originals (See To...,719.0,"{'Personal Success': 128, 'Personal Developmen..."
...,...,...,...,...,...,...,...,...,...,...
1850,"Zero Day: John Puller, Book 1",David Baldacci,4.4,4921.0,323.0,John Puller is a former war hero and now the b...,13 hours and 7 minutes,"[1,971 in Audible Audiobooks & Originals (See ...",1971.0,"{'International Mystery & Crime': 11, 'War & M..."
1851,Zero Limits: The Secret Hawaiian System for We...,Joe Vitale,4.4,886.0,703.0,If it seems like you work hard but never get ...,6 hours and 21 minutes,"[1,387 in Audible Audiobooks & Originals (See ...",1387.0,"{'Small Businesses': 16, 'Business Motivation ..."
1852,Zero to One,Peter Thiel,4.4,5306.0,615.0,What valuable company is nobody building? The ...,4 hours and 50 minutes,[25 in Audible Audiobooks & Originals (See Top...,25.0,"{'Business Management': 1, 'Entrepreneurship':..."
1853,Zindagi Aais Pais,Nikhil Sachan,4.1,198.0,501.0,"Zindagi Aais Pais, a short story collection, i...",4 hours and 4 minutes,"[3,026 in Audible Audiobooks & Originals (See ...",3026.0,"{'Fiction Short Stories': 35, 'Short Stories':..."


Unpacking genres and its ranks

In [604]:
df_exploded = df_merged.copy()

df_exploded['genre_ranks_list'] = df_exploded['genre_ranks'].apply(lambda x: x.items())
df_exploded = df_exploded.explode('genre_ranks_list')
df_exploded[['Genre', 'Genre_Rank']] = pd.DataFrame(
    df_exploded['genre_ranks_list'].tolist(),
    index=df_exploded.index
)

`df_exploded` has duplicate book rows with different genre and genre_rank values.

In [605]:
df_exploded

,Book Name,Author_x,Rating_x,Number of Reviews_x,Price_x,Description,Listening Time,Ranks and Genre,Audible_Rank,genre_ranks,genre_ranks_list,Genre,Genre_Rank
0,10 Essential Success Mantras from the Bhagavad...,Vimla Patil,4.2,45.0,233.0,"For thousands of years, the Bhagavad Gita has ...",26 minutes,"[3,076 in Audible Audiobooks & Originals (See ...",3076.0,"{'Hinduism': 45, 'Forecasting & Strategic Plan...","(Hinduism, 45)",Hinduism,45.0
0,10 Essential Success Mantras from the Bhagavad...,Vimla Patil,4.2,45.0,233.0,"For thousands of years, the Bhagavad Gita has ...",26 minutes,"[3,076 in Audible Audiobooks & Originals (See ...",3076.0,"{'Hinduism': 45, 'Forecasting & Strategic Plan...","(Forecasting & Strategic Planning, 57)",Forecasting & Strategic Planning,57.0
0,10 Essential Success Mantras from the Bhagavad...,Vimla Patil,4.2,45.0,233.0,"For thousands of years, the Bhagavad Gita has ...",26 minutes,"[3,076 in Audible Audiobooks & Originals (See ...",3076.0,"{'Hinduism': 45, 'Forecasting & Strategic Plan...","(Leadership, 136)",Leadership,136.0
1,10 Masterpieces You Have to Read Before You Die 1,Jane Austen,4.4,331.0,401.0,This Audiobook contains the following works :,102 hours and 19 minutes,"[1,485 in Audible Audiobooks & Originals (See ...",1485.0,"{'Caribbean & Latin American Literature': 4, '...","(Caribbean & Latin American Literature, 4)",Caribbean & Latin American Literature,4.0
1,10 Masterpieces You Have to Read Before You Die 1,Jane Austen,4.4,331.0,401.0,This Audiobook contains the following works :,102 hours and 19 minutes,"[1,485 in Audible Audiobooks & Originals (See ...",1485.0,"{'Caribbean & Latin American Literature': 4, '...","(English, 6)",English,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1852,Zero to One,Peter Thiel,4.4,5306.0,615.0,What valuable company is nobody building? The ...,4 hours and 50 minutes,[25 in Audible Audiobooks & Originals (See Top...,25.0,"{'Business Management': 1, 'Entrepreneurship':...","(Organisational Behaviour, 2)",Organisational Behaviour,2.0
1853,Zindagi Aais Pais,Nikhil Sachan,4.1,198.0,501.0,"Zindagi Aais Pais, a short story collection, i...",4 hours and 4 minutes,"[3,026 in Audible Audiobooks & Originals (See ...",3026.0,"{'Fiction Short Stories': 35, 'Short Stories':...","(Fiction Short Stories, 35)",Fiction Short Stories,35.0
1853,Zindagi Aais Pais,Nikhil Sachan,4.1,198.0,501.0,"Zindagi Aais Pais, a short story collection, i...",4 hours and 4 minutes,"[3,026 in Audible Audiobooks & Originals (See ...",3026.0,"{'Fiction Short Stories': 35, 'Short Stories':...","(Short Stories, 1539)",Short Stories,1539.0
1854,Zog,Julia Donaldson,4.9,2251.0,190.0,Zog is the keenest dragon in school. He's also...,28 minutes,"[1,966 in Audible Audiobooks & Originals (See ...",1966.0,"{'Fantasy': 1217, 'Children's Literature & Fic...","(Fantasy, 1217)",Fantasy,1217.0


### Creating a unique identifier for each book

In [606]:
# Removing columns with unhashable datatype
df_exploded.drop(columns=['Ranks and Genre', 'genre_ranks', 'genre_ranks_list'], inplace=True)

exclude_cols = ['Genre', 'Genre_Rank']
book_cols = [col for col in df_exploded.columns if col not in exclude_cols]
df_exploded['Book_ID'] = df_exploded.groupby(book_cols).ngroup() + 1

### Renaming and Re-Arranging columns

In [607]:
df_exploded.rename(columns={
    'Book Name':'Book_Name',
    'Author_x':'Author',
    'Rating_x':'Rating',
    'Number of Reviews_x':'Number_of_Reviews',
    'Price_x':'Price',
    'Listening Time':'Listening_Time'
    },
    inplace=True
)

df_filled = df_exploded.copy()
df_filled = df_filled[['Book_ID','Book_Name','Author','Rating','Number_of_Reviews','Price','Description','Listening_Time','Audible_Rank','Genre','Genre_Rank']]

In [608]:
# Percentage of missing values
per_miss_val(df_filled)

Book_ID              0.000000
Book_Name            0.000000
Author               0.000000
Rating               0.000000
Number_of_Reviews    0.000000
Price                0.000000
Description          0.000000
Listening_Time       0.000000
Audible_Rank         0.000000
Genre                0.039936
Genre_Rank           0.039936
dtype: float64

Removing rows with missing genre

In [609]:
df_filled = df_filled[~df_filled['Genre'].isnull()]

In [610]:
df_filled.describe(include='all')

,Book_ID,Book_Name,Author,Rating,Number_of_Reviews,Price,Description,Listening_Time,Audible_Rank,Genre,Genre_Rank
count,5006.000000,5006,5006,5006.000000,5006.000000,5006.000000,5006,5006,5006.000000,5006,5006.000000
unique,NaN,1833,1366,NaN,NaN,NaN,1699,966,NaN,854,NaN
top,NaN,Zero to One,Devdutt Pattanaik,NaN,NaN,NaN,Brought to you by Penguin.,11 hours and 4 minutes,NaN,Personal Success,NaN
freq,NaN,3,51,NaN,NaN,NaN,160,26,NaN,166,NaN
mean,924.416101,NaN,NaN,4.470276,1411.467639,1078.934279,NaN,NaN,2310.249101,NaN,1355.339992
std,529.715516,NaN,NaN,0.318813,3640.496342,1946.651508,NaN,NaN,3168.309614,NaN,10072.556656
min,1.000000,NaN,NaN,1.000000,1.000000,0.000000,NaN,NaN,-1.000000,NaN,1.000000
25%,468.000000,NaN,NaN,4.400000,98.250000,585.000000,NaN,NaN,662.000000,NaN,6.000000
50%,931.000000,NaN,NaN,4.500000,372.000000,703.000000,NaN,NaN,1981.000000,NaN,27.000000
75%,1386.000000,NaN,NaN,4.600000,1189.000000,938.000000,NaN,NaN,3321.750000,NaN,213.750000


## Data Formatting

In [611]:
df_filled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5006 entries, 0 to 1854
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Book_ID            5006 non-null   int64  
 1   Book_Name          5006 non-null   object 
 2   Author             5006 non-null   object 
 3   Rating             5006 non-null   float64
 4   Number_of_Reviews  5006 non-null   float64
 5   Price              5006 non-null   float64
 6   Description        5006 non-null   object 
 7   Listening_Time     5006 non-null   object 
 8   Audible_Rank       5006 non-null   float64
 9   Genre              5006 non-null   object 
 10  Genre_Rank         5006 non-null   float64
dtypes: float64(5), int64(1), object(5)
memory usage: 469.3+ KB


### Converting `Listening_Time` from object to timedelta

In [612]:
def extract_hours_minutes(text):
    hours = 0
    minutes = 0

    h = re.search(r'(\d+)\s*hour', text)
    m = re.search(r'(\d+)\s*minute', text)

    if h:
        hours = int(h.group(1))
    if m:
        minutes = int(m.group(1))
    
    return pd.Timedelta(hours=hours, minutes=minutes)

df_filled['Listening_Time'] = df_filled['Listening_Time'].apply(lambda x: extract_hours_minutes(x))

In [613]:
df_filled

,Book_ID,Book_Name,Author,Rating,Number_of_Reviews,Price,Description,Listening_Time,Audible_Rank,Genre,Genre_Rank
0,1,10 Essential Success Mantras from the Bhagavad...,Vimla Patil,4.2,45.0,233.0,"For thousands of years, the Bhagavad Gita has ...",0 days 00:26:00,3076.0,Hinduism,45.0
0,1,10 Essential Success Mantras from the Bhagavad...,Vimla Patil,4.2,45.0,233.0,"For thousands of years, the Bhagavad Gita has ...",0 days 00:26:00,3076.0,Forecasting & Strategic Planning,57.0
0,1,10 Essential Success Mantras from the Bhagavad...,Vimla Patil,4.2,45.0,233.0,"For thousands of years, the Bhagavad Gita has ...",0 days 00:26:00,3076.0,Leadership,136.0
1,2,10 Masterpieces You Have to Read Before You Die 1,Jane Austen,4.4,331.0,401.0,This Audiobook contains the following works :,4 days 06:19:00,1485.0,Caribbean & Latin American Literature,4.0
1,2,10 Masterpieces You Have to Read Before You Die 1,Jane Austen,4.4,331.0,401.0,This Audiobook contains the following works :,4 days 06:19:00,1485.0,English,6.0
...,...,...,...,...,...,...,...,...,...,...,...
1852,1833,Zero to One,Peter Thiel,4.4,5306.0,615.0,What valuable company is nobody building? The ...,0 days 04:50:00,25.0,Organisational Behaviour,2.0
1853,1834,Zindagi Aais Pais,Nikhil Sachan,4.1,198.0,501.0,"Zindagi Aais Pais, a short story collection, i...",0 days 04:04:00,3026.0,Fiction Short Stories,35.0
1853,1834,Zindagi Aais Pais,Nikhil Sachan,4.1,198.0,501.0,"Zindagi Aais Pais, a short story collection, i...",0 days 04:04:00,3026.0,Short Stories,1539.0
1854,1835,Zog,Julia Donaldson,4.9,2251.0,190.0,Zog is the keenest dragon in school. He's also...,0 days 00:28:00,1966.0,Fantasy,1217.0


In [614]:
genre_ranks_df = df_filled[['Book_ID', 'Genre', 'Genre_Rank', 'Rating']]
genre_ranks_df.reset_index(inplace=True)
genre_ranks_df.drop(columns=['index'], inplace=True)

C:\Users\jstha\AppData\Local\Temp\ipykernel_8488\390098799.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genre_ranks_df.drop(columns=['index'], inplace=True)


In [615]:
book_df = df_filled.drop(columns=['Genre', 'Genre_Rank'])
book_df.drop_duplicates(inplace=True)

In [616]:
book_df

,Book_ID,Book_Name,Author,Rating,Number_of_Reviews,Price,Description,Listening_Time,Audible_Rank
0,1,10 Essential Success Mantras from the Bhagavad...,Vimla Patil,4.2,45.0,233.0,"For thousands of years, the Bhagavad Gita has ...",0 days 00:26:00,3076.0
1,2,10 Masterpieces You Have to Read Before You Die 1,Jane Austen,4.4,331.0,401.0,This Audiobook contains the following works :,4 days 06:19:00,1485.0
2,3,10 Minutes 38 Seconds in This Strange World,Elif Shafak,4.5,507.0,752.0,Shortlisted for the Booker Prize 2019,0 days 09:11:00,221.0
3,4,100 Baggers: Stocks That Return 100-to-1 and H...,Christopher W. Mayer,4.5,178.0,668.0,This book is about 100-baggers. These are stoc...,0 days 06:19:00,2004.0
4,5,1000+ Little Things Happy Successful People Do...,Marc Chernoff,4.7,210.0,1005.0,New York Times best-selling authors Marc and A...,0 days 08:50:00,719.0
...,...,...,...,...,...,...,...,...,...
1850,1831,"Zero Day: John Puller, Book 1",David Baldacci,4.4,4921.0,323.0,John Puller is a former war hero and now the b...,0 days 13:07:00,1971.0
1851,1832,Zero Limits: The Secret Hawaiian System for We...,Joe Vitale,4.4,886.0,703.0,If it seems like you work hard but never get ...,0 days 06:21:00,1387.0
1852,1833,Zero to One,Peter Thiel,4.4,5306.0,615.0,What valuable company is nobody building? The ...,0 days 04:50:00,25.0
1853,1834,Zindagi Aais Pais,Nikhil Sachan,4.1,198.0,501.0,"Zindagi Aais Pais, a short story collection, i...",0 days 04:04:00,3026.0


In [617]:
genre_ranks_df

,Book_ID,Genre,Genre_Rank,Rating
0,1,Hinduism,45.0,4.2
1,1,Forecasting & Strategic Planning,57.0,4.2
2,1,Leadership,136.0,4.2
3,2,Caribbean & Latin American Literature,4.0,4.4
4,2,English,6.0,4.4
...,...,...,...,...
5001,1833,Organisational Behaviour,2.0,4.4
5002,1834,Fiction Short Stories,35.0,4.1
5003,1834,Short Stories,1539.0,4.1
5004,1835,Fantasy,1217.0,4.9


In [619]:
genre_ranks_df

,Book_ID,Genre,Genre_Rank,Rating,standard_genre
0,1,Hinduism,45.0,4.2,Hinduism
1,1,Forecasting & Strategic Planning,57.0,4.2,Forecasting & Strategic Planning
2,1,Leadership,136.0,4.2,Leadership
3,2,Caribbean & Latin American Literature,4.0,4.4,Caribbean & Latin American Literature
4,2,English,6.0,4.4,English
...,...,...,...,...,...
5001,1833,Organisational Behaviour,2.0,4.4,Organisational Behaviour
5002,1834,Fiction Short Stories,35.0,4.1,Fiction Short Stories
5003,1834,Short Stories,1539.0,4.1,Short Stories
5004,1835,Fantasy,1217.0,4.9,Fantasy


# Exploratory Data Analysis

### Distribution of ratings across genres

In [622]:
genre_ratings = genre_ranks_df.groupby('Genre')['Rating'].unique().reset_index(name='Ratings')

def avg_rating(lst):
    sum = 0
    count = 0
    for i in range(len(lst)):
        sum += lst[i]
        count += 1
    return round((sum/count), 2)

genre_ratings['Avg_Rating'] = genre_ratings['Ratings'].apply(lambda x: avg_rating(x))
genre_ratings.sort_values(by='Avg_Rating', ascending=False)

,Genre,Ratings,Avg_Rating
774,Superhero Fiction for Teens,[5.0],5.00
384,German Language Learning,[5.0],5.00
72,Audiobooks on Flowers & Trees for Children,[5.0],5.00
773,Superhero Fantasy,[5.0],5.00
463,International Relations & Globalization,[5.0],5.00
...,...,...,...
18,Adult Collections & Anthologies,"[3.9, 3.6, 3.7, 3.1, 3.5, 2.5, 3.0]",3.33
487,Language,[3.0],3.00
636,Plays,[3.0],3.00
368,French Language Learning,[3.0],3.00
